# CN7031 – Flight Delay 2024 | Big Data Analysis (PySpark)

This notebook runs the **whole** analysis pipeline with PySpark:

1. Load the data (sample or full 1.3 GB file)
2. Understand the dataset (schema, missing values, stats)
3. Clean it (missing / duplicates / outliers / cancelled)
4. Build extra features (season, day-part, distance band, delay bands ...)
5. **Descriptive analysis + lots of charts** (airline, airport, route, month,
   day, hour, season, distance, weekend, red-eye, state, causes, cancellations)
6. **Deeper analysis** (distributions, percentiles, correlations, delay
   propagation, two-way heatmaps, hub/network)
7. **Machine learning** (predict if a flight will be delayed + predict the
   delay minutes)
8. **Clustering** (group airports into performance segments)

All charts are saved to `outputs/figures/`, all tables to `outputs/tables/`.
Interactive **maps** (HTML) are saved to `outputs/figures/` too.

> Tip: keep `USE_SAMPLE = True` to test fast. Set it to `False` for the real
> full-dataset run.

## 0. Setup

In [ ]:
# make sure we can import the src package (run from the project root OR notebooks/)
import sys, os
sys.path.append(os.path.abspath(".."))   # if the notebook is inside notebooks/
sys.path.append(os.path.abspath("."))    # if run from the project root

import pandas as pd
from pyspark.sql import functions as F

from src import (config, data_loader, cleaning, features,
                 analysis, advanced_analysis, modeling, clustering,
                 visualize, maps)

USE_SAMPLE = True   # <-- set to False for the full 1.3 GB run
pd.set_option("display.max_columns", 50)

In [ ]:
# small helper: save an aggregated pandas table to outputs/tables as CSV
def save_table(pdf, name):
    path = config.TABLE_DIR / f"{name}.csv"
    pdf.to_csv(path, index=False)
    print("saved ->", path)
    return pdf

### Start Spark
`config.get_spark()` builds the session. **Keep this** – it contains the manual
fix for the old *"Could not initialize class ByteArrayMethods"* crash (the Java
`--add-opens` flags + Java-17 detection).

In [ ]:
spark = config.get_spark(memory="8g")
spark

## 1. Load the data

In [ ]:
df_raw = data_loader.load_flights(spark, sample=USE_SAMPLE)
print("rows:", df_raw.count(), "| columns:", len(df_raw.columns))
df_raw.printSchema()

In [ ]:
df_raw.show(5)

## 2. Dataset understanding
Before cleaning we look at the data: how many missing values, basic statistics,
and how many duplicate rows there are.

In [ ]:
# 2.1 missing values per column
missing = cleaning.missing_value_report(df_raw).toPandas().T
missing.columns = ["null_count"]
missing = missing.sort_values("null_count", ascending=False)
save_table(missing.reset_index().rename(columns={"index":"column"}), "missing_values")
missing.head(15)

In [ ]:
# 2.2 quick statistics of the main numeric columns
df_raw.select("dep_delay","arr_delay","distance","air_time","taxi_out").describe().show()

In [ ]:
# 2.3 how many duplicate rows?
total = df_raw.count()
distinct = df_raw.dropDuplicates().count()
print(f"total={total:,}  distinct={distinct:,}  duplicates={total-distinct:,}")

## 3. Cleaning
We clean step by step and print the row count after every step, so we can see
exactly how many rows each rule removes.

In [ ]:
steps = [
    ("start",               df_raw),
    ("parse dates",         cleaning.parse_dates(df_raw)),
]
d = cleaning.parse_dates(df_raw)
d = cleaning.drop_duplicates(d);            steps.append(("drop duplicates", d))
d = cleaning.drop_cancelled_and_diverted(d);steps.append(("drop cancelled/diverted", d))
d = cleaning.drop_missing_delays(d);        steps.append(("drop missing delays", d))
d = cleaning.remove_delay_outliers(d);      steps.append(("remove outliers", d))

rows = [(name, df.count()) for name, df in steps]
clean_log = pd.DataFrame(rows, columns=["step","rows"])
clean_log["removed"] = clean_log["rows"].shift(1) - clean_log["rows"]
save_table(clean_log, "cleaning_steps")
clean_log

In [ ]:
df_clean = cleaning.clean_flights(df_raw)
print("clean rows:", df_clean.count())

## 4. Feature engineering
Add helpful new columns (delay flags, delay bands, dep hour, day-part, season,
weekend, red-eye, route, distance band, cause flags).

In [ ]:
df = features.add_features(df_clean)
# cache because we reuse this dataframe many times below
df = df.cache()
df.count()
print("columns now:", len(df.columns))
df.select("op_unique_carrier","arr_delay","is_delayed","delay_band",
          "dep_hour","day_part","season","is_weekend","distance_band","route").show(5)

## 5. Experiments, results and visualisation (descriptive)

### 5.1 Delays by airline

In [ ]:
by_airline = analysis.delays_by_airline(df).toPandas()
save_table(by_airline, "delays_by_airline")
visualize.bar_chart(by_airline, "op_unique_carrier", "avg_arr_delay",
    "Average arrival delay by airline", "Airline", "Avg arrival delay (min)",
    "delay_by_airline.png")
visualize.grouped_bar(by_airline.head(12), "op_unique_carrier",
    ["avg_dep_delay","avg_arr_delay"], "Departure vs arrival delay by airline",
    "Airline", "Minutes", "airline_dep_vs_arr.png", labels=["dep","arr"])
by_airline.head(10)

### 5.2 Delays by origin airport + airport bubble MAP

In [ ]:
by_airport = analysis.delays_by_origin_airport(df, min_flights=200).toPandas()
save_table(by_airport, "delays_by_airport")
visualize.hbar_chart(by_airport.sort_values("flights", ascending=False),
    "origin", "flights", "Busiest origin airports", "Airport", "Flights",
    "delay_by_airport.png", top=20)
# interactive map: bubble size = flights, colour = avg delay
maps.airport_bubble_map(by_airport, code_col="origin")
by_airport.head(10)

### 5.3 Delays by destination airport

In [ ]:
by_dest = analysis.delays_by_dest_airport(df, min_flights=200).toPandas()
save_table(by_dest, "delays_by_dest")
visualize.lollipop_chart(by_dest.sort_values("avg_arr_delay", ascending=False).head(15),
    "dest", "avg_arr_delay", "Worst destination airports (avg arrival delay)",
    "Airport", "Avg arrival delay (min)", "worst_dest_airports.png")
by_dest.head(10)

### 5.4 Busiest routes + route MAP

In [ ]:
by_route = analysis.delays_by_route(df, min_flights=100).toPandas()
save_table(by_route, "delays_by_route")
busiest = advanced_analysis.busiest_routes_volume(df, top=40).toPandas()
save_table(busiest, "busiest_routes")
visualize.hbar_chart(busiest, "route", "flights",
    "Busiest routes by number of flights", "Route", "Flights",
    "busiest_routes.png", top=20)
maps.route_map(busiest)
busiest.head(10)

### 5.5 Delays by month (seasonal trend)

In [ ]:
by_month = analysis.delays_by_month(df).toPandas()
save_table(by_month, "delays_by_month")
visualize.line_chart(by_month, "month", "avg_arr_delay",
    "Average arrival delay by month", "Month", "Avg arrival delay (min)",
    "delay_by_month.png")
visualize.dual_axis(by_month, "month", "flights", "avg_arr_delay",
    "Flights vs average delay by month", "Flights", "Avg delay (min)",
    "month_flights_vs_delay.png")
by_month

### 5.6 Delays by day of week

In [ ]:
by_dow = analysis.delays_by_day_of_week(df).toPandas()
save_table(by_dow, "delays_by_day_of_week")
visualize.bar_chart(by_dow, "day_of_week", "avg_arr_delay",
    "Average arrival delay by day of week (1=Mon ... 7=Sun)",
    "Day of week", "Avg arrival delay (min)", "delay_by_day_of_week.png",
    color="#81b29a")
by_dow

### 5.7 Delays by hour of day

In [ ]:
by_hour = analysis.delays_by_hour(df).toPandas()
save_table(by_hour, "delays_by_hour")
visualize.line_chart(by_hour, "dep_hour", "avg_arr_delay",
    "Average arrival delay by scheduled departure hour", "Hour (0-23)",
    "Avg arrival delay (min)", "delay_by_hour.png")
by_hour

### 5.8 Delays by season

In [ ]:
by_season = analysis.delays_by_season(df).toPandas()
save_table(by_season, "delays_by_season")
visualize.bar_chart(by_season, "season", "avg_arr_delay",
    "Average arrival delay by season", "Season", "Avg arrival delay (min)",
    "delay_by_season.png", color="#3d405b")
by_season

### 5.9 Delays by part of the day

In [ ]:
by_part = analysis.delays_by_day_part(df).toPandas()
save_table(by_part, "delays_by_day_part")
visualize.bar_chart(by_part, "day_part", "delayed_pct",
    "% of flights delayed by part of day", "Part of day", "% delayed",
    "delay_by_day_part.png", color="#e07a5f")
by_part

### 5.10 Delays by distance band (short vs long haul)

In [ ]:
by_dist = analysis.delays_by_distance_band(df).toPandas()
save_table(by_dist, "delays_by_distance_band")
visualize.bar_chart(by_dist, "distance_band", "avg_arr_delay",
    "Average arrival delay by trip length", "Distance band",
    "Avg arrival delay (min)", "delay_by_distance.png", color="#588157")
by_dist

### 5.11 Weekend vs weekday  &  5.12 Red-eye vs daytime

In [ ]:
wk = analysis.delays_weekend_vs_weekday(df).toPandas()
save_table(wk, "delays_weekend_vs_weekday")
re = analysis.delays_by_redeye(df).toPandas()
save_table(re, "delays_by_redeye")
print(wk[["kind","flights","avg_arr_delay","delayed_pct"]])
print(re[["kind","flights","avg_arr_delay","delayed_pct"]])

### 5.13 Delays by US state (choropleth MAP)

In [ ]:
by_state = analysis.delays_by_state(df).toPandas()
save_table(by_state, "delays_by_state")
maps.state_choropleth(by_state, state_col="origin_state_nm", value_col="avg_arr_delay")
by_state.head(10)

### 5.14 Delay bands (how late, in buckets)

In [ ]:
bands = analysis.delay_band_counts(df).toPandas()
save_table(bands, "delay_band_counts")
visualize.pie_chart(bands, "delay_band", "flights",
    "Share of flights by delay band", "delay_bands_pie.png")
bands

### 5.15 What causes the delays?

In [ ]:
# total minutes per cause
causes = analysis.delay_cause_totals(df).toPandas().T.reset_index()
causes.columns = ["cause","minutes"]
save_table(causes, "delay_causes")
visualize.pie_chart(causes, "cause", "minutes",
    "Share of total delay minutes by cause", "delay_causes_pie.png")

# how often each cause appears (% of flights)
freq = analysis.delay_cause_frequency(df).toPandas().T.reset_index()
freq.columns = ["cause","pct_of_flights"]
save_table(freq, "delay_cause_frequency")

# cause mix by month (stacked)
cmonth = analysis.cause_share_by_month(df).toPandas()
save_table(cmonth, "cause_share_by_month")
visualize.stacked_bar(cmonth, "month",
    ["carrier_delay","weather_delay","nas_delay","security_delay","late_aircraft_delay"],
    "Delay-cause minutes by month", "Month", "Total minutes",
    "cause_by_month_stacked.png",
    labels=["carrier","weather","nas","security","late aircraft"])
freq

### 5.16 Cancellations and diversions (uses the RAW data)

In [ ]:
overview = analysis.cancellation_overview(df_raw).toPandas()
save_table(overview, "cancellation_overview")
by_reason = analysis.cancellations_by_reason(df_raw).toPandas()
save_table(by_reason, "cancellations_by_reason")
canc_air = analysis.cancellations_by_airline(df_raw).toPandas()
save_table(canc_air, "cancellations_by_airline")
print(overview)
visualize.bar_chart(by_reason, "cancellation_code", "cancellations",
    "Cancellations by reason (A=carrier B=weather C=NAS D=security)",
    "Reason code", "Cancellations", "cancellations_by_reason.png", color="#bc4749")
by_reason

## 6. Deeper / statistical analysis

### 6.1 Distribution & percentiles of arrival delay

In [ ]:
desc = advanced_analysis.delay_describe(df).toPandas()
save_table(desc, "delay_describe")
pct = advanced_analysis.delay_percentiles(df).toPandas()
save_table(pct, "delay_percentiles")
hist = advanced_analysis.delay_histogram(df).toPandas()
visualize.histogram(hist, "bin", "flights",
    "Distribution of arrival delay (minutes)", "Arrival delay (min)",
    "Flights", "delay_histogram.png")
print(desc); print(pct)

In [ ]:
# boxplot of delay by season (uses a sample of raw rows)
season_sample = df.select("season","arr_delay").sample(False, 0.2, seed=1).toPandas()
visualize.boxplot_by_group(season_sample, "season", "arr_delay",
    "Arrival delay spread by season", "delay_boxplot_season.png")

### 6.2 Correlation between numeric columns

In [ ]:
corr = advanced_analysis.correlation_matrix(df)
save_table(corr.reset_index(), "correlation_matrix")
visualize.correlation_heatmap(corr, "Correlation of numeric flight columns",
    "correlation_heatmap.png")
corr.round(2)

### 6.3 Delay propagation: does leaving late mean arriving late?

In [ ]:
scat = advanced_analysis.propagation_scatter_sample(df, 5000)
visualize.scatter(scat, "dep_delay", "arr_delay",
    "Departure delay vs arrival delay", "Departure delay (min)",
    "Arrival delay (min)", "propagation_scatter.png")
prop = advanced_analysis.propagation_by_depdelay_band(df).toPandas()
save_table(prop, "propagation_by_band")
prop

### 6.4 Two-way heatmaps

In [ ]:
hd = advanced_analysis.heatmap_hour_vs_dow(df)
visualize.heatmap(hd, "Avg arrival delay by hour (rows) and day of week (cols)",
    "heatmap_hour_dow.png", xlabel="Day of week", ylabel="Hour")
am = advanced_analysis.heatmap_airline_vs_month(df)
visualize.heatmap(am, "Avg arrival delay by airline (rows) and month (cols)",
    "heatmap_airline_month.png", xlabel="Month", ylabel="Airline")

### 6.5 Hub / network analysis

In [ ]:
traffic = advanced_analysis.airport_traffic(df, top=20).toPandas()
save_table(traffic, "airport_traffic")
visualize.hbar_chart(traffic, "airport", "total_movements",
    "Busiest airports by total movements (dep + arr)", "Airport",
    "Movements", "airport_traffic.png", top=20)
traffic.head(10)

### 6.6 Airline scorecard + radar

In [ ]:
score = advanced_analysis.airline_scorecard(df, min_flights=1000).toPandas()
save_table(score, "airline_scorecard")
visualize.radar_chart(score.sort_values("on_time_pct", ascending=False),
    "op_unique_carrier", ["on_time_pct","avg_arr_delay","p90_delay","big_delay_pct"],
    "Top airlines compared (normalised)", "airline_radar.png", top=5)
score

## 7. Predictive modelling (Spark MLlib)
We only use information known **before** the flight leaves (airline, origin,
dest, month, day, hour, distance, season, day-part) – no cheating with columns
that are only known after landing.

> On the full dataset this is the slow part. If it takes too long, train on a
> sub-sample with `prepare_model_data(df, max_rows=500000)`.

In [ ]:
model_data = modeling.prepare_model_data(df)   # add max_rows=... to subsample
model_data = model_data.cache()
print("rows used for modelling:", model_data.count())

### 7.1 Classification – will the flight be delayed (15+ min)?

In [ ]:
clf = modeling.train_classifiers(model_data)
metrics = pd.DataFrame(clf["metrics"]).T
save_table(metrics.reset_index().rename(columns={"index":"model"}), "model_metrics")
visualize.bar_chart(metrics.reset_index(), "index", "AUC",
    "Model comparison (AUC, higher = better)", "Model", "AUC",
    "model_auc.png", color="#9d4edd")
metrics

In [ ]:
# confusion matrix + feature importance for the Random Forest
cm = modeling.confusion_counts(clf["models"]["RandomForest"], clf["test"])
print(cm)
imp = modeling.feature_importance(clf["models"]["RandomForest"], top=15)
save_table(imp, "feature_importance")
visualize.hbar_chart(imp, "feature", "importance",
    "Random Forest feature importance", "Feature", "Importance",
    "feature_importance.png", top=15)
imp

### 7.2 Regression – how many minutes late?

In [ ]:
reg = modeling.train_regressor(model_data)
print("Regression metrics:", reg["metrics"])
pd.DataFrame([reg["metrics"]])

## 8. Clustering – grouping airports into performance segments
We build a profile for each airport (how busy, average delay, % delayed, taxi-out
time, % long-haul) and let K-Means find natural groups.

In [ ]:
profiles = clustering.build_airport_profiles(df, min_flights=1000).cache()
print("airports profiled:", profiles.count())
k_table = clustering.choose_k(profiles, k_values=(2,3,4,5,6))
save_table(k_table, "kmeans_silhouette")
visualize.line_chart(k_table, "k", "silhouette",
    "Choosing k (higher silhouette = better)", "k (clusters)", "silhouette",
    "kmeans_silhouette.png")
k_table

In [ ]:
clustered = clustering.cluster_airports(profiles, k=4)
save_table(clustered, "airport_clusters")
summary = clustering.cluster_profiles_summary(clustered)
save_table(summary, "cluster_summary")
visualize.cluster_scatter(clustered, "flights", "avg_arr_delay", "cluster",
    "Airport clusters: traffic vs average delay", "Flights (per year)",
    "Avg arrival delay (min)", "airport_clusters.png")
summary

## 9. Stop Spark

In [ ]:
spark.stop()
print("done - figures in outputs/figures, tables in outputs/tables")